## Problem

User input is messy.

Numbers may contain leading/trailing whitespace, thousands separators, invalid characters, negative values when only positive numbers are allowed, or values outside an acceptable range.

Your task is to implement a function that safely parses a numeric string into a `Decimal`, while enforcing configurable validation rules and providing descriptive error messages.

---

## Requirements

Implement the following function:

```python
parse_numeric_input(
    raw: str,
    *,
    allow_negative: bool = True,
    max_value: Optional[float] = None,
) -> Decimal
```

The function must return a `Decimal` representing the parsed value.

---

### Validation Rules

| Rule                 | Behaviour                                            |
| -------------------- | ---------------------------------------------------- |
| Input type           | `raw` must be a `str`; otherwise raise `TypeError`   |
| Whitespace           | Ignore leading and trailing whitespace               |
| Thousands separators | Ignore commas (`,`), e.g. `"1,234.56"`               |
| Parsing              | Invalid numeric strings must raise `ValueError`      |
| Finite numbers       | Reject `NaN`, `Infinity`, and `-Infinity`            |
| Negative values      | Reject negative values when `allow_negative=False`   |
| Maximum value        | Reject values greater than `max_value` when provided |

---

## Examples

```python
parse_numeric_input("42")
# Decimal('42')

parse_numeric_input("   123.45   ")
# Decimal('123.45')

parse_numeric_input("1,234,567.89")
# Decimal('1234567.89')

parse_numeric_input("-15", allow_negative=False)
# ValueError: Negative values are not allowed, got -15.

parse_numeric_input("500", max_value=100)
# ValueError: Value 500 exceeds maximum allowed 100.

parse_numeric_input("hello")
# ValueError: Cannot parse 'hello' as a number.

parse_numeric_input(float("nan"))
# TypeError: Expected str, got float
```

---

## Constraints

* Use only the Python standard library.
* Use `decimal.Decimal` for parsing.
* Do **not** use `float()` during parsing.
* Remove commas before parsing.
* Reject `NaN` and infinite values.
* `max_value` is optional.
* Return a `Decimal` object.

---

## Hints

* **Hint 1 — Why `Decimal` instead of `float`?**

  `float` cannot exactly represent many decimal values.

  ```python
  >>> 0.1 + 0.2
  0.30000000000000004
  ```

  `Decimal` preserves decimal precision and is commonly used for financial software.

---

* **Hint 2 — `Decimal` accepts special values**

  ```python
  Decimal("NaN")
  Decimal("Infinity")
  ```

  These parse successfully, so you must explicitly reject them.

---

* **Hint 3 — Thousands separators aren't valid**

  ```python
  Decimal("1,000")
  ```

  raises an exception.

  Remove commas before attempting to parse.

---

* **Hint 4 — Comparing Decimals**

  If a maximum value is supplied as a `float`, convert it safely:

  ```python
  Decimal(str(max_value))
  ```

  rather than

  ```python
  Decimal(max_value)
  ```

---

## 80/20 Insight

Most production input-validation bugs come from trusting user input too early.

A robust parser should:

* validate the input type,
* normalize formatting,
* parse using an exact numeric representation,
* reject non-finite values,
* enforce business rules (such as positivity and limits),
* and fail with clear, descriptive exceptions.

This validation pipeline covers the vast majority of numeric-input scenarios encountered in APIs, CLIs, financial applications, and configuration systems.

---

# Solution


In [ ]:
from decimal import Decimal, InvalidOperation
from typing import Optional

In [ ]:
def parse_numeric_input(raw: str, *, allow_negative: bool = True, max_value: Optional[float] = None) -> Decimal:
    """
    Parses a numeric string into a Decimal, applying validation rules.
    Raises ValueError with descriptive messages on invalid input.
    """
    if not isinstance(raw, str):
        raise TypeError(f"Expected str, got {type(raw).__name__}")
    
    cleaned = raw.strip().replace(',', '')  # handle locale-style separators

    try:
        value = Decimal(cleaned)
    except InvalidOperation:
        raise ValueError(f"Cannot parse {raw!r} as a number.")
    
    if value.is_nan() or value.is_infinite():
        raise ValueError(f"Value must be finite, got {raw!r}.")
    
    if not allow_negative and value < 0:
        raise ValueError(f"Negative values are not allowed, got {value}.")
    
    if max_value is not None and value > Decimal(str(max_value)):
        raise ValueError(f"Value {value} exceeds maximum allowed {max_value}.")
    
    return value

In [ ]:
print(parse_numeric_input("1,234.56"))  # Should return Decimal('1234.56')
print(parse_numeric_input("-42.4"))  # Should return Decimal('-42.4')
# parse_numeric_input("NaN")  # Should raise ValueError
# parse_numeric_input("inf")  # Should raise ValueError